In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [ ]:
seed = 1 # not used
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20

In [ ]:
data = pd.read_parquet("../data/GBG500.parquet")
data

In [ ]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

In [ ]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [ ]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

In [ ]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

In [ ]:
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values

In [ ]:
spectral_energy = np.nansum(X_feat, axis=1)
anomaly_scores = spectral_energy

ap = average_precision_score(y_true, anomaly_scores)
print(f"Total spectral energy AP = {ap:.4f}")
sb.glue("GBG500_ap_spectral_energy", float(ap))

ranks = pd.Series(anomaly_scores).rank(ascending=False, method='first').astype(int)
labels_sorted[labels_sorted["label"] == "Reckless"].assign(rank=ranks)[["ride_id", "rank", "label"]].sort_values("rank")

In [ ]:
ap_random_theoretical = y_true.mean()
print(f"Theoretical random AP = {ap_random_theoretical:.4f}")
sb.glue("GBG500_ap_random", float(ap_random_theoretical))